In [ ]:
import fastf1
import requests
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

In [ ]:
schedule = fastf1.get_event_schedule(2025)
schedule

In [ ]:
# Which round comes next
NEXT_GP = 6

def get_sessions(NEXT_GP):
    all_data = []
    for round in range(1, NEXT_GP):
        for session_type in ['FP1', 'FP2', 'FP3', 'Q', 'R']:
            try:
                session = fastf1.get_session(2025, round, session_type)
                session.load()
                all_data.append(session)
    
            except Exception as e:
                print(f"Skipped Round {round} {session_type}: {e}")
    return all_data

In [ ]:
all_data = get_sessions(NEXT_GP)

In [ ]:
results = pd.DataFrame()
rounds = []

for rnd in range(len(all_data)):
    rnd_no = int(all_data[rnd].event.RoundNumber)
    session_info = all_data[rnd].session_info['Meeting']['OfficialName']
    session_name = all_data[rnd].name

    if rnd_no not in rounds:
        
        if session_name == 'Practice 1':
            
            data = all_data[rnd].laps[['Driver', 'DriverNumber', 'Team', 'LapTime', 'Compound', 'SpeedFL', 'SpeedST']].copy()
            data = data.sort_values('LapTime', ascending=True)
            data = data.drop_duplicates(subset='Driver', keep='first')
            row_data_list = []
            for _, row in data.iterrows():
                row_data_list.append({
                    'Round': rnd_no,
                    'Event': session_info,
                    'Driver': row['Driver'],
                    'DriverNumber': row['DriverNumber'],
                    'Team': row['Team'],
                    'FP1_LapTime': row['LapTime'],
                    'FP1_Compound': row['Compound'],
                    'FP1_SpeedFL': row['SpeedFL'],
                    'FP1_SpeedST': row['SpeedST'],
                })
            results = pd.concat([results, pd.DataFrame(row_data_list)], ignore_index=True)
                
        if session_name =='Practice 2':
            
            data = all_data[rnd].laps[['Driver', 'DriverNumber', 'Team', 'LapTime', 'Compound', 'SpeedFL', 'SpeedST']].copy()
            data = data.sort_values('LapTime', ascending=True)
            data = data.drop_duplicates(subset='Driver', keep='first')
            for _, row in data.iterrows():
                results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 
                           ['FP2_LapTime', 'FP2_Compound', 'FP2_SpeedFL', 'FP2_SpeedST']] = [
                               row['LapTime'], row['Compound'], row['SpeedFL'], row['SpeedST']
                           ]
        if session_name =='Practice 3':
            
            data = all_data[rnd].laps[['Driver', 'DriverNumber', 'Team', 'LapTime', 'Compound', 'SpeedFL', 'SpeedST']].copy()
            data = data.sort_values('LapTime', ascending=True)
            data = data.drop_duplicates(subset='Driver', keep='first')
            for _, row in data.iterrows():
                results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 
                           ['FP3_LapTime', 'FP3_Compound', 'FP3_SpeedFL', 'FP3_SpeedST']] = [
                               row['LapTime'], row['Compound'], row['SpeedFL'], row['SpeedST']
                           ]                   
    
        if session_name =='Qualifying':
    
            data = all_data[rnd].results[['DriverNumber','Q1','Q2','Q3']].copy()
            data = data.sort_values('Q3')
            for _, row in data.iterrows():
                results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), ['Q1','Q2','Q3']] = [row['Q1'], row['Q2'], row['Q3']]
        
        if session_name =='Race':
            data = all_data[rnd].laps[['DriverNumber','LapTime']].copy()
            data = data.sort_values('LapTime', ascending=True)
            data = data.drop_duplicates(subset='DriverNumber', keep='first')
            points = all_data[rnd].results[['DriverNumber','Points']]
    
            if 'Total_Points' not in results.columns:
                results['Total_Points'] = 0.0
                
            for _, row in data.iterrows():
                results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 'FastestRaceLap'] = row['LapTime']
    
            for _, row in points.iterrows():
                results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 
                    'Total_Points'] += row['Points']
            rounds.append(rnd_no)